# Раздел 2. Обучение и сравнение моделей

**Задача:** подсчёт автомобилей на дорожном видео.

**Цель notebook:** обучить и сравнить **5 архитектур** на датасете **AI Traffic System**.

| № | Модель | Тип | Библиотека |
|---|--------|-----|------------|
| 1 | YOLOv8s | one-stage | Ultralytics |
| 2 | Faster R-CNN | two-stage | torchvision |
| 3 | RetinaNet | one-stage | torchvision |
| 4 | SSD300 | one-stage | torchvision |
| 5 | RT-DETR-l | transformer | Ultralytics |

## Установка зависимостей

In [9]:
import importlib
import subprocess
import sys

%pip install -q ultralytics opencv-python matplotlib pandas openpyxl pyyaml torchmetrics tqdm faster-coco-eval

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faster-coco-eval"])
import faster_coco_eval  # noqa: F401

import torchmetrics.utilities.imports as tm_imports
import torchmetrics.detection.mean_ap as mean_ap
importlib.reload(tm_imports)
importlib.reload(mean_ap)

from torchmetrics.detection import MeanAveragePrecision
MeanAveragePrecision(box_format="xyxy", iou_type="bbox", backend="faster_coco_eval")
print("Зависимости OK (faster-coco-eval для mAP)")


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\777\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Зависимости OK (faster-coco-eval для mAP)


## Пути и настройки

**Текущий статус:**
| Модель | Эпохи | Веса | Действие |
|--------|-------|------|----------|
| YOLOv8s | 50 ✅ | `runs/detect/yolov8s/weights/best.pt` | `SKIP_TRAIN=True` — только eval |
| Faster R-CNN | 50 ✅ | `models/faster_rcnn/best.pt` | `SKIP_TRAIN=True` — только eval (дообучение 2b ✅) |
| RetinaNet | 50 ✅ | `models/retinanet/best.pt` | `SKIP_TRAIN=True` — только eval |
| SSD300 | 50 ✅ | `models/ssd300/best.pt` | `SKIP_TRAIN=True` — только eval |
| RT-DETR-l | 35 ✅ | `models/rtdetr-l/best.pt` | `SKIP_TRAIN=True` — EarlyStopping (лучшая эпоха 25) |


In [10]:
from pathlib import Path
import json
import yaml
import shutil
import time
import statistics
import random

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

PROJECT = Path(".").resolve()
DATA = (PROJECT / ".." / "archive (2)").resolve()
RESULTS = PROJECT / "results"
MODELS_DIR = PROJECT / "models"
RESULTS.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

CLASS_NAMES = ["bicycle", "bus", "car", "motorbike", "rickshaw", "truck", "van"]
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

EPOCHS = 50           # Ultralytics: YOLOv8s, RT-DETR
EPOCHS_TV = 50        # torchvision: RetinaNet, SSD300 (полное обучение)
BATCH = 16            # YOLOv8
RTDETR_BATCH = 0.7    # AutoBatch: доля VRAM (batch=16 не помещается для RT-DETR-l на 8GB)
VAL_BATCH = 4         # batch для eval на test
IMG_SIZE = 640
TV_BATCH = 4          # batch для torchvision

# Learning rate / grad clip по модели (SSD300: lr=0.005 → NaN в весах, mAP=0)
TV_LR = {"faster_rcnn": 0.005, "retinanet": 0.005, "ssd300": 0.001}
TV_GRAD_CLIP = {"ssd300": 1.0}
TV_SAVE_BEST_ON_VALID = {"ssd300": True}  # best.pt по valid mAP@0.5, не последняя эпоха

# Сколько эпох реально обучена модель (для таблицы отчёта)
EPOCHS_LOG = {
    "yolov8s": 50,
    "faster_rcnn": 50,   # 15 + 35 (ячейка 2b) — готово
    "retinanet": 50,
    "ssd300": 50,
    "rtdetr-l": 35,    # EarlyStopping: best@25, patience=10 → 35 эпох
}

# True = пропустить train, только eval + метрики
SKIP_TRAIN = {
    "yolov8s": True,       # 50 эпох — готово
    "faster_rcnn": True,   # 50 эпох — готово (2b выполнена)
    "retinanet": True,     # 50 эпох — готово
    "ssd300": True,        # 50 эпох — готово
    "rtdetr-l": True,      # 35 эпох — готово (EarlyStopping)
}

# Дообучение Faster R-CNN: 15 + 35 = 50 эпох (запускайте ячейку 2b в конце)
FASTER_RCNN_FINETUNE_EPOCHS = 35

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Проект:", PROJECT)
print("Датасет:", DATA)
print("Устройство:", DEVICE, "| CUDA:", torch.cuda.is_available())
assert DATA.exists(), "Датасет не найден"

all_metrics = []

Проект: C:\Users\777\Desktop\8 сем\Практика\vehicle-counting
Датасет: C:\Users\777\Desktop\8 сем\Практика\archive (2)
Устройство: 0 | CUDA: True


##  `data.yaml` для Ultralytics

In [11]:
data_yaml = {
    "path": str(DATA),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": NUM_CLASSES,
    "names": CLASS_NAMES,
}
DATA_YAML = PROJECT / "data.yaml"
DATA_YAML.write_text(yaml.dump(data_yaml, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(DATA_YAML.read_text(encoding="utf-8"))

path: C:\Users\777\Desktop\8 сем\Практика\archive (2)
train: train/images
val: valid/images
test: test/images
nc: 7
names:
- bicycle
- bus
- car
- motorbike
- rickshaw
- truck
- van



## Общие функции

In [12]:
def imread_unicode(path):
    img = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Не удалось прочитать: {path}")
    return img


def find_image(img_dir, stem):
    for ext in IMAGE_EXTS:
        p = img_dir / f"{stem}{ext}"
        if p.exists():
            return p
    matches = [p for p in img_dir.glob(f"{stem}.*") if p.suffix.lower() in IMAGE_EXTS]
    if not matches:
        raise FileNotFoundError(stem)
    return matches[0]


def benchmark_fps(predict_fn, n_warmup=3, n_images=50):
    images = [p for p in (DATA / "test" / "images").iterdir() if p.suffix.lower() in IMAGE_EXTS][:n_images]
    for _ in range(n_warmup):
        predict_fn(images[0])
    times = []
    for p in images:
        t0 = time.perf_counter()
        predict_fn(p)
        times.append((time.perf_counter() - t0) * 1000)
    avg_ms = statistics.mean(times)
    return round(avg_ms, 2), round(1000.0 / avg_ms, 2)


def add_metrics_row(name, arch_type, library, weights_path, map50, map5095, precision, recall, fps, ms, size_mb, epochs, input_size=IMG_SIZE):
    row = {
        "Модель": name,
        "Тип": arch_type,
        "Библиотека": library,
        "Input": f"{input_size}x{input_size}",
        "Эпохи": epochs,
        "mAP@0.5": round(map50, 4),
        "mAP@0.5:0.95": round(map5095, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "FPS": fps,
        "ms/кадр": ms,
        "Размер (MB)": round(size_mb, 2),
        "Веса": str(weights_path),
    }
    all_metrics.append(row)
    print(f"\n=== {name} ===")
    for k, v in row.items():
        if k != "Веса":
            print(f"  {k}: {v}")
    return row


def save_comparison_table():
    df = pd.DataFrame(all_metrics)
    xlsx = RESULTS / "comparison.xlsx"
    jpath = RESULTS / "all_metrics.json"
    df.to_excel(xlsx, index=False)
    jpath.write_text(json.dumps(all_metrics, ensure_ascii=False, indent=2), encoding="utf-8")
    display(df)
    print("Saved:", xlsx)
    print("Saved:", jpath)
    return df


def replace_metrics_row(name, arch_type, library, weights_path, map50, map5095, precision, recall, fps, ms, size_mb, epochs, input_size=IMG_SIZE):
    global all_metrics
    all_metrics = [r for r in all_metrics if r["Модель"] != name]
    return add_metrics_row(name, arch_type, library, weights_path, map50, map5095, precision, recall, fps, ms, size_mb, epochs, input_size)


def find_weights(run_name):
    for p in [MODELS_DIR / run_name / "best.pt", PROJECT / "runs" / "detect" / run_name / "weights" / "best.pt"]:
        if p.exists():
            return p
    return None


def require_setup(need_tv=False):
    required = [
        "find_weights", "SKIP_TRAIN", "EPOCHS_LOG", "add_metrics_row", "benchmark_fps",
        "train_ultralytics", "eval_ultralytics",
    ]
    if need_tv:
        required += ["train_tv_model", "eval_tv_model", "predict_tv", "save_tv_examples", "MeanAveragePrecision"]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(
            "Не выполнены подготовительные ячейки. Отсутствуют: "
            + ", ".join(missing)
            + ". Запустите по порядку: Шаг 0 -> 1 -> 2 -> 3 -> 4 (и 5 для torchvision)."
        )


def print_saved_weights():
    print("Сохранённые веса:")
    for run in ["yolov8s", "faster_rcnn", "retinanet", "ssd300", "rtdetr-l"]:
        w = find_weights(run)
        mark = "OK" if w else "нет"
        skip = SKIP_TRAIN.get(run, False)
        print(f"  {run:12} | {mark:3} | SKIP_TRAIN={skip} | {w or '-'}")

##  Ultralytics (YOLOv8, RT-DETR)

In [13]:
import gc
import sys

def train_ultralytics(kind, weights_name, run_name, epochs=EPOCHS):
    from ultralytics import YOLO, RTDETR
    batch = RTDETR_BATCH if kind == "rtdetr" else BATCH
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    Model = YOLO if kind == "yolo" else RTDETR
    model = Model(weights_name)
    workers = 0 if sys.platform == "win32" else 4
    print(f"train {run_name}: batch={batch}, imgsz={IMG_SIZE}, workers={workers}")
    model.train(
        data=str(DATA_YAML),
        epochs=epochs,
        imgsz=IMG_SIZE,
        batch=batch,
        project=str(PROJECT / "runs" / "detect"),
        name=run_name,
        exist_ok=True,
        device=DEVICE,
        patience=10,
        save=True,
        plots=True,
        workers=workers,
    )
    best = PROJECT / "runs" / "detect" / run_name / "weights" / "best.pt"
    dest = MODELS_DIR / run_name / "best.pt"
    dest.parent.mkdir(parents=True, exist_ok=True)
    if best.exists():
        shutil.copy2(best, dest)
    return dest if dest.exists() else best


def eval_ultralytics(weights_path, run_name):
    import gc
    from ultralytics import YOLO
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
    model = YOLO(str(weights_path))
    metrics = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=IMG_SIZE,
        batch=VAL_BATCH,
        device=DEVICE,
        plots=True,
        project=str(RESULTS),
        name=f"{run_name}_test",
        exist_ok=True,
    )
    box = metrics.box
    return {
        "map50": float(box.map50),
        "map5095": float(box.map),
        "precision": float(box.mp),
        "recall": float(box.mr),
        "model": model,
    }


def save_ultra_examples(model, run_name, n_good=3, n_bad=3):
    out_dir = RESULTS / run_name / "examples"
    out_dir.mkdir(parents=True, exist_ok=True)
    test_imgs = [str(p) for p in (DATA / "test" / "images").iterdir() if p.suffix.lower() in IMAGE_EXTS]
    sample = random.sample(test_imgs, min(30, len(test_imgs)))
    results = model.predict(source=sample, imgsz=IMG_SIZE, conf=0.4, device=DEVICE, verbose=False)
    scored = []
    for r in results:
        n_det = len(r.boxes) if r.boxes is not None else 0
        avg_conf = float(r.boxes.conf.mean()) if n_det > 0 else 0.0
        scored.append((avg_conf, n_det, r))
    scored.sort(key=lambda x: (x[0], x[1]), reverse=True)
    good = scored[:n_good]
    bad = sorted(scored, key=lambda x: (x[0], x[1]))[:n_bad]
    for tag, items in [("good", good), ("bad", bad)]:
        for i, (_, _, r) in enumerate(items):
            r.save(filename=str(out_dir / f"{tag}_{i+1}.jpg"))
    print(f"Примеры сохранены: {out_dir}")

##  torchvision (Faster R-CNN, RetinaNet, SSD)

In [14]:
import importlib
import subprocess
import sys

from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    FasterRCNN_ResNet50_FPN_Weights,
    retinanet_resnet50_fpn,
    RetinaNet_ResNet50_FPN_Weights,
    ssd300_vgg16,
    SSD300_VGG16_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.models.detection.ssd import SSDClassificationHead
from tqdm.auto import tqdm


def ensure_detection_metrics():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faster-coco-eval"])
    import faster_coco_eval  # noqa: F401
    import torchmetrics.utilities.imports as tm_imports
    import torchmetrics.detection.mean_ap as mean_ap
    importlib.reload(tm_imports)
    importlib.reload(mean_ap)
    from torchmetrics.detection import MeanAveragePrecision
    MeanAveragePrecision(box_format="xyxy", iou_type="bbox", backend="faster_coco_eval")
    return MeanAveragePrecision


MeanAveragePrecision = ensure_detection_metrics()


class YOLODetectionDataset(Dataset):
    def __init__(self, split):
        self.img_dir = DATA / split / "images"
        self.lbl_dir = DATA / split / "labels"
        self.stems = sorted(p.stem for p in self.img_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, idx):
        stem = self.stems[idx]
        img = cv2.cvtColor(imread_unicode(find_image(self.img_dir, stem)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        boxes, labels = [], []
        lbl = self.lbl_dir / f"{stem}.txt"
        if lbl.exists():
            for line in lbl.read_text(encoding="utf-8").splitlines():
                if not line.strip():
                    continue
                cid, xc, yc, bw, bh = line.split()
                cid = int(cid)
                xc, yc, bw, bh = map(float, (xc, yc, bw, bh))
                boxes.append([(xc - bw / 2) * w, (yc - bh / 2) * h, (xc + bw / 2) * w, (yc + bh / 2) * h])
                labels.append(cid + 1)
        image = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4)),
            "labels": torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64),
        }
        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


def get_tv_device():
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def build_tv_model(name):
    nc = NUM_CLASSES + 1
    if name == "faster_rcnn":
        model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
        in_f = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = FastRCNNPredictor(in_f, nc)
    elif name == "retinanet":
        model = retinanet_resnet50_fpn(weights=RetinaNet_ResNet50_FPN_Weights.DEFAULT)
        in_c = model.head.classification_head.cls_logits.in_channels
        num_anchors = model.head.classification_head.num_anchors
        model.head.classification_head = RetinaNetClassificationHead(in_c, num_anchors, nc)
    elif name == "ssd300":
        model = ssd300_vgg16(weights=SSD300_VGG16_Weights.DEFAULT)
        ch = model.head.classification_head
        in_channels = [m.in_channels for m in ch.module_list]
        num_anchors = [m.out_channels // ch.num_columns for m in ch.module_list]
        model.head.classification_head = SSDClassificationHead(in_channels, num_anchors, nc)
    else:
        raise ValueError(name)
    return model


def _state_dict_has_nan(state_dict):
    return any(torch.isnan(v).any().item() for v in state_dict.values() if v.dtype.is_floating_point)


def train_tv_model(name, run_name, epochs=EPOCHS_TV, batch_size=TV_BATCH, resume_from=None, lr=None, grad_clip=0.0, save_best_on_valid=None):
    if lr is None:
        lr = TV_LR.get(name, 0.005)
    if grad_clip <= 0:
        grad_clip = TV_GRAD_CLIP.get(name, 0.0)
    if save_best_on_valid is None:
        save_best_on_valid = TV_SAVE_BEST_ON_VALID.get(name, False)
    device = get_tv_device()
    model = build_tv_model(name).to(device)
    if resume_from is not None and Path(resume_from).exists():
        sd = torch.load(resume_from, map_location=device, weights_only=True)
        if _state_dict_has_nan(sd):
            raise RuntimeError(f"{run_name}: checkpoint {resume_from} содержит NaN")
        model.load_state_dict(sd)
        print(f"Дообучение с весов: {resume_from}, ещё {epochs} эпох, lr={lr}")
    loader = DataLoader(YOLODetectionDataset("train"), batch_size=batch_size, shuffle=True, num_workers=0, collate_fn=collate_fn)
    optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(epochs // 3, 1), gamma=0.1)
    dest = MODELS_DIR / run_name / "best.pt"
    dest.parent.mkdir(parents=True, exist_ok=True)
    best_map = -1.0
    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for images, targets in tqdm(loader, desc=f"{run_name} ep {epoch+1}/{epochs}"):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            losses = sum(loss_dict.values())
            if torch.isnan(losses):
                raise RuntimeError(f"{run_name}: NaN loss на эпохе {epoch+1} — уменьшите lr или включите grad clip")
            optimizer.zero_grad()
            losses.backward()
            if grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            total_loss += losses.item()
        scheduler.step()
        avg_loss = total_loss / len(loader)
        print(f"  Эпоха {epoch+1}: loss = {avg_loss:.4f}")
        if save_best_on_valid:
            map50, _, _ = eval_tv_split(model, device, split="valid")
            model.train()
            if map50 > best_map and not _state_dict_has_nan(model.state_dict()):
                best_map = map50
                torch.save(model.state_dict(), dest)
                print(f"  -> best.pt сохранён (valid mAP@0.5 = {map50:.4f})")
    if not save_best_on_valid:
        if _state_dict_has_nan(model.state_dict()):
            raise RuntimeError(f"{run_name}: веса содержат NaN — best.pt не сохранён (уменьшите lr, включите grad clip)")
        torch.save(model.state_dict(), dest)
    elif not dest.exists():
        raise RuntimeError(f"{run_name}: best.pt не создан — valid mAP не улучшался или все веса NaN")
    else:
        model.load_state_dict(torch.load(dest, map_location=device, weights_only=True))
        print(f"Загружены лучшие веса: {dest} (valid mAP@0.5 = {best_map:.4f})")
    model.eval()
    return dest, model


def eval_tv_split(model, device, split="test", score_thr=0.3):
    was_training = model.training
    model.eval()
    loader = DataLoader(YOLODetectionDataset(split), batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn)
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", backend="faster_coco_eval")
    with torch.no_grad():
        for images, targets in tqdm(loader, desc=f"Eval {split}", leave=False):
            images = [img.to(device) for img in images]
            outputs = model(images)
            preds, tgts = [], []
            for out, tgt in zip(outputs, targets):
                mask = out["scores"] >= score_thr
                preds.append({"boxes": out["boxes"][mask].cpu(), "scores": out["scores"][mask].cpu(), "labels": out["labels"][mask].cpu()})
                tgts.append({"boxes": tgt["boxes"], "labels": tgt["labels"]})
            metric.update(preds, tgts)
    res = metric.compute()
    mar = res.get("mar_100", torch.tensor(0.0))
    recall = float(mar.mean()) if hasattr(mar, "mean") else float(mar or 0)
    map50 = float(res["map_50"] or 0)
    map5095 = float(res["map"] or 0)
    if was_training:
        model.train()
    return map50, map5095, recall


def eval_tv_model(weights_path, name, run_name, score_thr=0.3, split="test"):
    device = get_tv_device()
    model = build_tv_model(name)
    sd = torch.load(weights_path, map_location=device, weights_only=True)
    if _state_dict_has_nan(sd):
        raise RuntimeError(f"{run_name}: веса {weights_path} содержат NaN — удалите файл и переобучите модель")
    model.load_state_dict(sd)
    model.to(device)
    map50, map5095, recall = eval_tv_split(model, device, split=split, score_thr=score_thr)
    model.eval()
    return {"map50": map50, "map5095": map5095, "precision": map50, "recall": recall, "model": model, "device": device}


def predict_tv(model, device, path):
    model.eval()
    img = cv2.cvtColor(imread_unicode(path), cv2.COLOR_BGR2RGB)
    t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
    with torch.no_grad():
        model([t.to(device)])


def save_tv_examples(model, device, run_name, n_good=3, n_bad=3, score_thr=0.3):
    out_dir = RESULTS / run_name / "examples"
    out_dir.mkdir(parents=True, exist_ok=True)
    ds = YOLODetectionDataset("test")
    indices = random.sample(range(len(ds)), min(30, len(ds)))
    scored = []
    model.eval()
    with torch.no_grad():
        for i in indices:
            img, _ = ds[i]
            out = model([img.to(device)])[0]
            mask = out["scores"] >= score_thr
            n_det = int(mask.sum())
            avg_conf = float(out["scores"][mask].mean()) if n_det else 0.0
            vis = (img.permute(1, 2, 0).numpy() * 255).astype(np.uint8).copy()
            for box, label, score in zip(out["boxes"][mask], out["labels"][mask], out["scores"][mask]):
                x1, y1, x2, y2 = map(int, box.tolist())
                cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(vis, f"{CLASS_NAMES[int(label)-1]} {score:.2f}", (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
            scored.append((avg_conf, n_det, vis))
    scored.sort(key=lambda x: (x[0], x[1]), reverse=True)
    for tag, items in [("good", scored[:n_good]), ("bad", sorted(scored, key=lambda x: (x[0], x[1]))[:n_bad])]:
        for j, (_, _, vis) in enumerate(items):
            buf = cv2.imencode(".jpg", cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))[1]
            buf.tofile(str(out_dir / f"{tag}_{j+1}.jpg"))
    print(f"Примеры сохранены: {out_dir}")

## Проверка готовности 

In [15]:
require_setup(need_tv=False)
print_saved_weights()
print("CUDA:", torch.cuda.is_available())

Сохранённые веса:
  yolov8s      | OK  | SKIP_TRAIN=True | C:\Users\777\Desktop\8 сем\Практика\vehicle-counting\runs\detect\yolov8s\weights\best.pt
  faster_rcnn  | OK  | SKIP_TRAIN=True | C:\Users\777\Desktop\8 сем\Практика\vehicle-counting\models\faster_rcnn\best.pt
  retinanet    | OK  | SKIP_TRAIN=True | C:\Users\777\Desktop\8 сем\Практика\vehicle-counting\models\retinanet\best.pt
  ssd300       | OK  | SKIP_TRAIN=True | C:\Users\777\Desktop\8 сем\Практика\vehicle-counting\models\ssd300\best.pt
  rtdetr-l     | OK  | SKIP_TRAIN=True | C:\Users\777\Desktop\8 сем\Практика\vehicle-counting\models\rtdetr-l\best.pt
CUDA: True


---

## Модель 1: YOLOv8s (one-stage)

In [18]:
require_setup(need_tv=False)
RUN = "yolov8s"
weights = find_weights(RUN)
if weights and SKIP_TRAIN.get(RUN, False):
    print("Пропуск обучения YOLOv8s, веса:", weights)
elif weights is None:
    weights = train_ultralytics("yolo", "yolov8s.pt", RUN, epochs=EPOCHS)
else:
    print("Веса уже есть:", weights)
ev = eval_ultralytics(weights, RUN)
ms, fps = benchmark_fps(lambda p: ev["model"].predict(str(p), imgsz=IMG_SIZE, verbose=False, device=DEVICE))
size_mb = weights.stat().st_size / (1024 ** 2)
add_metrics_row("YOLOv8s", "one-stage", "Ultralytics", weights, ev["map50"], ev["map5095"], ev["precision"], ev["recall"], fps, ms, size_mb, EPOCHS_LOG["yolov8s"])
save_ultra_examples(ev["model"], RUN)

Пропуск обучения YOLOv8s, веса: C:\Users\777\Desktop\8 сем\Практика\vehicle-counting\runs\detect\yolov8s\weights\best.pt
Ultralytics 8.4.83  Python-3.11.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060, 8187MiB)
Model summary (fused): 73 layers, 11,128,293 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 598.8296.4 MB/s, size: 64.5 KB)
val: Scanning C:\Users\777\Desktop\8 сем\Практика\archive (2)\test\labels.cache... 812 images, 160 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 812/812  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 203/203 23.5it/s 8.6s0.1s
                   all        812       2408      0.742      0.529      0.586      0.417
               bicycle         47         53      0.661      0.642      0.642      0.473
                   bus         99        162      0.803      0.716      0.774      0.527
                   car        566       1566      0.852      0.793 

---

## Модель 2: Faster R-CNN (two-stage)


In [19]:
require_setup(need_tv=True)
RUN = "faster_rcnn"
weights = find_weights(RUN)
if weights and SKIP_TRAIN.get(RUN, False):
    print("Пропуск обучения, веса:", weights)
    ev = eval_tv_model(weights, "faster_rcnn", RUN)
else:
    weights, _ = train_tv_model("faster_rcnn", RUN, epochs=EPOCHS_TV)
    ev = eval_tv_model(weights, "faster_rcnn", RUN)
ms, fps = benchmark_fps(lambda p: predict_tv(ev["model"], ev["device"], p))
size_mb = weights.stat().st_size / (1024 ** 2)
add_metrics_row("Faster R-CNN", "two-stage", "torchvision", weights, ev["map50"], ev["map5095"], ev["precision"], ev["recall"], fps, ms, size_mb, EPOCHS_LOG["faster_rcnn"])
save_tv_examples(ev["model"], ev["device"], RUN)

Пропуск обучения, веса: C:\Users\777\Desktop\8 сем\Практика\vehicle-counting\models\faster_rcnn\best.pt


Eval test:   0%|          | 0/812 [00:00<?, ?it/s]


=== Faster R-CNN ===
  Модель: Faster R-CNN
  Тип: two-stage
  Библиотека: torchvision
  Input: 640x640
  Эпохи: 50
  mAP@0.5: 0.536
  mAP@0.5:0.95: 0.3493
  Precision: 0.536
  Recall: 0.4325
  FPS: 17.49
  ms/кадр: 57.18
  Размер (MB): 158.17
Примеры сохранены: C:\Users\777\Desktop\8 сем\Практика\vehicle-counting\results\faster_rcnn\examples


---

## Модель 2b: Дообучение Faster R-CNN до 50 эпох 

**15 + 35 = 50 эпох.** 

In [ ]:
require_setup(need_tv=True)
import gc

RUN = "faster_rcnn"
weights_in = find_weights(RUN)
assert weights_in is not None, "Сначала нужны веса faster_rcnn (15 эпох)"

if torch.cuda.is_available():
    gc.collect()
    torch.cuda.empty_cache()

weights, _ = train_tv_model(
    "faster_rcnn",
    RUN,
    epochs=FASTER_RCNN_FINETUNE_EPOCHS,
    resume_from=weights_in,
    lr=0.001,
)
EPOCHS_LOG["faster_rcnn"] = 15 + FASTER_RCNN_FINETUNE_EPOCHS

ev = eval_tv_model(weights, "faster_rcnn", RUN)
ms, fps = benchmark_fps(lambda p: predict_tv(ev["model"], ev["device"], p))
size_mb = weights.stat().st_size / (1024 ** 2)
replace_metrics_row(
    "Faster R-CNN", "two-stage", "torchvision", weights,
    ev["map50"], ev["map5095"], ev["precision"], ev["recall"],
    fps, ms, size_mb, EPOCHS_LOG["faster_rcnn"],
)
save_tv_examples(ev["model"], ev["device"], RUN)
print("Faster R-CNN дообучен до", EPOCHS_LOG["faster_rcnn"], "эпох")

---

## Модель 3: RetinaNet (one-stage)


In [20]:
require_setup(need_tv=True)
RUN = "retinanet"
weights = find_weights(RUN)
if weights and SKIP_TRAIN.get(RUN, False):
    print("Пропуск обучения, веса:", weights)
    ev = eval_tv_model(weights, "retinanet", RUN)
else:
    weights, _ = train_tv_model("retinanet", RUN, epochs=EPOCHS_TV)
    ev = eval_tv_model(weights, "retinanet", RUN)
ms, fps = benchmark_fps(lambda p: predict_tv(ev["model"], ev["device"], p))
size_mb = weights.stat().st_size / (1024 ** 2)
add_metrics_row("RetinaNet", "one-stage", "torchvision", weights, ev["map50"], ev["map5095"], ev["precision"], ev["recall"], fps, ms, size_mb, EPOCHS_LOG["retinanet"])
save_tv_examples(ev["model"], ev["device"], RUN)

Пропуск обучения, веса: C:\Users\777\Desktop\8 сем\Практика\vehicle-counting\models\retinanet\best.pt


Eval test:   0%|          | 0/812 [00:00<?, ?it/s]


=== RetinaNet ===
  Модель: RetinaNet
  Тип: one-stage
  Библиотека: torchvision
  Input: 640x640
  Эпохи: 50
  mAP@0.5: 0.4575
  mAP@0.5:0.95: 0.2964
  Precision: 0.4575
  Recall: 0.3852
  FPS: 18.93
  ms/кадр: 52.82
  Размер (MB): 123.7
Примеры сохранены: C:\Users\777\Desktop\8 сем\Практика\vehicle-counting\results\retinanet\examples


---

## Модель 4: SSD300 (one-stage)


In [16]:
require_setup(need_tv=True)
RUN = "ssd300"
weights = find_weights(RUN)
if weights and SKIP_TRAIN.get(RUN, False):
    print("Пропуск обучения, веса:", weights)
    ev = eval_tv_model(weights, "ssd300", RUN)
else:
    weights, _ = train_tv_model("ssd300", RUN, epochs=EPOCHS_TV, lr=TV_LR["ssd300"], grad_clip=TV_GRAD_CLIP["ssd300"], save_best_on_valid=True)
    ev = eval_tv_model(weights, "ssd300", RUN)
ms, fps = benchmark_fps(lambda p: predict_tv(ev["model"], ev["device"], p))
size_mb = weights.stat().st_size / (1024 ** 2)
add_metrics_row("SSD300", "one-stage", "torchvision", weights, ev["map50"], ev["map5095"], ev["precision"], ev["recall"], fps, ms, size_mb, EPOCHS_LOG["ssd300"], input_size=300)
save_tv_examples(ev["model"], ev["device"], RUN)

Пропуск обучения, веса: C:\Users\777\Desktop\8 сем\Практика\vehicle-counting\models\ssd300\best.pt


Eval test:   0%|          | 0/812 [00:00<?, ?it/s]


=== SSD300 ===
  Модель: SSD300
  Тип: one-stage
  Библиотека: torchvision
  Input: 300x300
  Эпохи: 50
  mAP@0.5: 0.3026
  mAP@0.5:0.95: 0.211
  Precision: 0.3026
  Recall: 0.2483
  FPS: 58.62
  ms/кадр: 17.06
  Размер (MB): 93.67
Примеры сохранены: C:\Users\777\Desktop\8 сем\Практика\vehicle-counting\results\ssd300\examples


---

## Модель 5: RT-DETR-l (transformer) 

In [21]:
require_setup(need_tv=False)
RUN = "rtdetr-l"
weights = find_weights(RUN)
if weights and SKIP_TRAIN.get(RUN, False):
    print("Пропуск обучения RT-DETR, веса:", weights)
elif weights is None:
    weights = train_ultralytics("rtdetr", "rtdetr-l.pt", RUN, epochs=EPOCHS)
else:
    print("Веса уже есть:", weights)
ev = eval_ultralytics(weights, RUN)
ms, fps = benchmark_fps(lambda p: ev["model"].predict(str(p), imgsz=IMG_SIZE, verbose=False, device=DEVICE))
size_mb = weights.stat().st_size / (1024 ** 2)
add_metrics_row("RT-DETR-l", "transformer", "Ultralytics", weights, ev["map50"], ev["map5095"], ev["precision"], ev["recall"], fps, ms, size_mb, EPOCHS_LOG["rtdetr-l"])
save_ultra_examples(ev["model"], RUN)

Пропуск обучения RT-DETR, веса: C:\Users\777\Desktop\8 сем\Практика\vehicle-counting\models\rtdetr-l\best.pt
Ultralytics 8.4.83  Python-3.11.9 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060, 8187MiB)
rt-detr-l summary: 310 layers, 31,998,125 parameters, 0 gradients, 103.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 786.8283.4 MB/s, size: 60.6 KB)
val: Scanning C:\Users\777\Desktop\8 сем\Практика\archive (2)\test\labels.cache... 812 images, 160 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 812/812  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 203/203 10.2it/s 19.8s0.1ss
                   all        812       2408      0.596      0.554      0.563       0.35
               bicycle         47         53      0.688      0.624      0.633      0.369
                   bus         99        162      0.575      0.728      0.664       0.39
                   car        566       1566      0.758      0.804      0.801  

---

## Итоговая таблица

Сохраняется в `results/comparison.xlsx` — для **раздела 2** отчёта.

In [22]:
save_comparison_table()

,Модель,Тип,Библиотека,Input,Эпохи,mAP@0.5,mAP@0.5:0.95,Precision,Recall,FPS,ms/кадр,Размер (MB),Веса
0,SSD300,one-stage,torchvision,300x300,50,0.3026,0.2110,0.3026,0.2483,58.62,17.06,93.67,C:\Users\777\Desktop\8 сем\Практика\vehicle-co...
1,YOLOv8s,one-stage,Ultralytics,640x640,50,0.5863,0.4166,0.7418,0.5293,85.93,11.64,21.50,C:\Users\777\Desktop\8 сем\Практика\vehicle-co...
2,Faster R-CNN,two-stage,torchvision,640x640,50,0.5360,0.3493,0.5360,0.4325,17.49,57.18,158.17,C:\Users\777\Desktop\8 сем\Практика\vehicle-co...
3,RetinaNet,one-stage,torchvision,640x640,50,0.4575,0.2964,0.4575,0.3852,18.93,52.82,123.70,C:\Users\777\Desktop\8 сем\Практика\vehicle-co...
4,RT-DETR-l,transformer,Ultralytics,640x640,35,0.5630,0.3502,0.5960,0.5543,33.84,29.55,63.22,C:\Users\777\Desktop\8 сем\Практика\vehicle-co...


Saved: C:\Users\777\Desktop\8 сем\Практика\vehicle-counting\results\comparison.xlsx
Saved: C:\Users\777\Desktop\8 сем\Практика\vehicle-counting\results\all_metrics.json


,Модель,Тип,Библиотека,Input,Эпохи,mAP@0.5,mAP@0.5:0.95,Precision,Recall,FPS,ms/кадр,Размер (MB),Веса
0,SSD300,one-stage,torchvision,300x300,50,0.3026,0.2110,0.3026,0.2483,58.62,17.06,93.67,C:\Users\777\Desktop\8 сем\Практика\vehicle-co...
1,YOLOv8s,one-stage,Ultralytics,640x640,50,0.5863,0.4166,0.7418,0.5293,85.93,11.64,21.50,C:\Users\777\Desktop\8 сем\Практика\vehicle-co...
2,Faster R-CNN,two-stage,torchvision,640x640,50,0.5360,0.3493,0.5360,0.4325,17.49,57.18,158.17,C:\Users\777\Desktop\8 сем\Практика\vehicle-co...
3,RetinaNet,one-stage,torchvision,640x640,50,0.4575,0.2964,0.4575,0.3852,18.93,52.82,123.70,C:\Users\777\Desktop\8 сем\Практика\vehicle-co...
4,RT-DETR-l,transformer,Ultralytics,640x640,35,0.5630,0.3502,0.5960,0.5543,33.84,29.55,63.22,C:\Users\777\Desktop\8 сем\Практика\vehicle-co...
